# Inflation forecasting: 2022–2025

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'modelling').is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from modelling.regime_experiment import notebook_result_table, plot_notebook_forecasts, plot_predicted_regimes

RESULTS_FILE = PROJECT_ROOT / 'data/processed/model_results/results_static_2022_2025.json'
TEST_START, TEST_END = '2022-01-01', '2025-11-01'
if not RESULTS_FILE.exists():
    raise FileNotFoundError('Run `python -m modelling.regime_experiment` before opening Notebook 2.')
results = json.loads(RESULTS_FILE.read_text(encoding='utf-8'))
expected_signature = {'test_start': TEST_START, 'test_end': TEST_END, 'version': 5, 'main_estimation': 'static'}
mismatches = {key: (results.get('signature', {}).get(key), value)
              for key, value in expected_signature.items()
              if results.get('signature', {}).get(key) != value}
if mismatches:
    raise ValueError(f'Results do not match Notebook 2 configuration: {mismatches}')
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
display(pd.Series(results['sample'], name='Current comparison').to_frame())

In [ ]:
def show_comparison(group):
    table = notebook_result_table(results, group)
    display(table.drop(columns='key').round(4))
    return table

def plot_rmse(table, title):
    ordered = table.sort_values('rmse', ascending=False)
    ax = ordered.plot.barh(x='model', y='rmse', legend=False, figsize=(12, max(3, .28 * len(ordered))))
    ax.set(xlabel='RMSE (percentage points)', ylabel='', title=title)
    ax.grid(axis='x', alpha=.2)
    plt.tight_layout()
    plt.show()

def matching_ar_gains(table):
    table = table.copy()
    ar = {tuple(results['models'][row.key]['features']): row.rmse
          for row in table.itertuples() if row.family == 'AR'}
    table['gain_vs_matching_AR_pct'] = [
        100 * (1 - row.rmse / ar[tuple(results['models'][row.key]['features'])])
        if row.family in ('AR-HMM', 'MS-Variance') else np.nan for row in table.itertuples()
    ]
    return table

## Static baselines

In [ ]:
static_table = show_comparison('Static baselines')
plot_rmse(static_table, 'Static model comparison')
plot_notebook_forecasts(results, static_table['key'], 'Static forecasts: 2022–2025')
plt.show()

## Expanding-window effect

In [ ]:
window_table = show_comparison('Static vs expanding')
plot_rmse(window_table, 'Coefficient estimation: static versus expanding')
plot_notebook_forecasts(results, window_table['key'], 'AR(1) and AR-HMM(1): static versus expanding coefficients')
plt.show()

## Static two-state AR-HMM: alternative switching structures
The number of states stays fixed at two. These specifications test whether regimes are better identified when the AR coefficient is common, when volatility switches, or when only volatility switches. Model coefficients are estimated once through December 2021; the charts use probabilities available before observing each test-month outcome. The preferred variance-only model labels its regimes Low volatility and High volatility because its conditional mean equation is common across states.

In [ ]:
structure_table = notebook_result_table(results, 'MoM two-state structures')
structure_rows = []
for row in structure_table.itertuples():
    model = results['models'][row.key]
    frame = pd.DataFrame(model['forecasts'])
    labels = model['parameters']['regime_labels']
    order = model['parameters']['raw_state_order']
    variances = np.asarray(model['parameters']['state_variances'])[order]
    shares = frame.PredictedRegime.value_counts(normalize=True).reindex(labels, fill_value=0)
    probabilities = frame[[f'Probability_{label}' for label in labels]]
    structure_rows.append({
        'model': model['name'], 'rmse': row.rmse, 'bic_training': model['parameters']['bic'],
        'level_1_share': shares.iloc[0], 'level_2_share': shares.iloc[1],
        'level_1_sd': np.sqrt(variances[0]), 'level_2_sd': np.sqrt(variances[1]),
        'mean_classification_confidence': probabilities.max(axis=1).mean(),
    })
structure_summary = pd.DataFrame(structure_rows).sort_values('rmse').reset_index(drop=True)
display(structure_summary.round(4))
plot_rmse(structure_table, 'Static two-state AR-HMM switching structures')
for key in results['groups']['MoM two-state structures']:
    model = results['models'][key]
    frame = pd.DataFrame(model['forecasts'])
    frame['Date'] = pd.to_datetime(frame['Date'])
    labels = model['parameters']['regime_labels']
    plot_predicted_regimes(frame, labels)
    plt.suptitle(model['name'], y=1.02)
    plt.show()

## Preferred specification for the remaining analysis
The two-state variance-switching model with a common intercept and common coefficients has the best training BIC, the lowest test RMSE, and uses both regimes. It is therefore used below for every lag, macroeconomic and sentiment comparison. The model is labelled **MS-Variance AR**: its states describe normal versus turbulent inflation volatility, not low versus high conditional mean inflation.

## Monthly inflation: static lag orders 1–6

In [ ]:
mom_lags = matching_ar_gains(notebook_result_table(results, 'MoM lag orders'))
display(mom_lags.drop(columns='key').round(4))
plot_rmse(mom_lags, 'Monthly inflation: static AR and two-state MS-Variance AR by lag order')
best_mom_keys = mom_lags.loc[mom_lags.family.isin(['AR', 'MS-Variance'])].sort_values('rmse').groupby('family').first()['key'].tolist()
plot_notebook_forecasts(results, best_mom_keys, 'Monthly inflation: lowest test RMSE in each family')
plt.show()

## MoM: static macroeconomic indicators with the preferred MS-Variance model

In [ ]:
macro_table = matching_ar_gains(notebook_result_table(results, 'MoM macro'))
display(macro_table.drop(columns='key').round(4))
plot_rmse(macro_table, 'MoM: all macro combinations')
best_keys = macro_table.sort_values('rmse').groupby('family').first()['key'].tolist()
plot_notebook_forecasts(results, best_keys, 'MoM: lowest macro test RMSE in each family')
plt.show()

## MoM: static sentiment with and without macro inputs using MS-Variance

In [ ]:
sentiment_table = show_comparison('MoM sentiment')
plot_rmse(sentiment_table, 'MoM: sentiment comparison')
plot_notebook_forecasts(results, sentiment_table['key'], 'MoM: baseline and selected macro models with sentiment')
plt.show()

## Monthly inflation: static alternative FinBERT aggregations using MS-Variance

In [ ]:
aggregation_table = matching_ar_gains(notebook_result_table(results, 'MoM sentiment aggregations'))
display(aggregation_table.drop(columns='key').round(4))
plot_rmse(aggregation_table, 'Monthly inflation: mean, median, negative share and dispersion')

## Year-over-year inflation: static lag orders 1–6

In [ ]:
yoy_lags = matching_ar_gains(notebook_result_table(results, 'YoY lag orders'))
display(yoy_lags.drop(columns='key').round(4))
plot_rmse(yoy_lags, 'YoY inflation: AR, two-state MS-Variance AR, and naive')
best_yoy_keys = yoy_lags.loc[yoy_lags.family.isin(['AR', 'MS-Variance', 'Naive'])].sort_values('rmse').groupby('family').first()['key'].tolist()
plot_notebook_forecasts(results, best_yoy_keys, 'YoY: lowest test RMSE in each family')
plt.show()

## YoY: static macroeconomic indicators with the preferred MS-Variance model

In [ ]:
macro_table = matching_ar_gains(notebook_result_table(results, 'YoY macro'))
display(macro_table.drop(columns='key').round(4))
plot_rmse(macro_table, 'YoY: all macro combinations')
best_keys = macro_table.sort_values('rmse').groupby('family').first()['key'].tolist()
plot_notebook_forecasts(results, best_keys, 'YoY: lowest macro test RMSE in each family')
plt.show()

## YoY: static sentiment with and without macro inputs using MS-Variance

In [ ]:
sentiment_table = show_comparison('YoY sentiment')
plot_rmse(sentiment_table, 'YoY: sentiment comparison')
plot_notebook_forecasts(results, sentiment_table['key'], 'YoY: baseline and selected macro models with sentiment')
plt.show()

## Overall static results

In [ ]:
main_groups = ['Static baselines', 'MoM lag orders', 'MoM macro', 'MoM sentiment',
               'MoM sentiment aggregations', 'YoY lag orders', 'YoY macro', 'YoY sentiment']
main_keys = {key for group in main_groups for key in results['groups'][group]}
for target in ['Inflation_MoM', 'Inflation_YoY']:
    display(Markdown(f'**{target}**'))
    overall = pd.DataFrame([{'model': results['models'][key]['name'],
                             'family': results['models'][key]['family'],
                             'estimation': results['models'][key]['mode'],
                             **results['models'][key]['metrics']}
                            for key in main_keys if results['models'][key]['target'] == target
                            and results['models'][key]['mode'] == 'static']).sort_values('rmse')
    display(overall.round(4))